# PyTorch basics lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ankitrgupta1/silicon-to-agents/blob/main/labs/pytorch-basics.ipynb)

The handful of PyTorch ideas that everything else in this repo uses: tensors, shapes, broadcasting, autograd, modules and the training loop. Companion to [Python & PyTorch](../prereqs/pytorch.md). Runs on CPU in seconds.

The `assert` lines are checks: if one fails, something isn't what the text says.

In [ ]:
try:
    import s2a
except ImportError:
    %pip install -q git+https://github.com/ankitrgupta1/silicon-to-agents

import os, time
import torch
import torch.nn as nn

STEPS = int(os.environ.get("S2A_STEPS", 500))
torch.manual_seed(0)
print("PyTorch", torch.__version__)

## 1. Tensors: arrays with a shape and a type

Most bugs are shape bugs. Get in the habit of printing `.shape`.

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print(x.shape, x.dtype)                 # 2 rows, 3 columns, 32-bit floats
print(x[0], x[:, 1], x.sum(dim=0))      # a row, a column, column sums
print(x.reshape(3, 2).shape, x.T.shape) # same 6 numbers, different shapes

ids = torch.tensor([3, 1, 4])           # whole numbers (token ids) are int64
assert x.dtype == torch.float32 and ids.dtype == torch.int64

## 2. Matrix multiply and broadcasting

`@` is matrix multiplication: `(n, d) @ (d, k)` gives `(n, k)`. For element-wise operations, PyTorch **broadcasts**: it lines shapes up from the right and stretches any dimension of size 1.

In [ ]:
W = torch.randn(3, 4)
print("matmul:", (x @ W).shape)                       # (2, 3) @ (3, 4) -> (2, 4)

col, row = torch.ones(3, 1), torch.arange(4.0)
print("broadcast:", (col + row).shape)                # (3, 1) + (4,) -> (3, 4)

batch = torch.randn(8, 16, 32)                        # (batch, tokens, width)
print("batched:", (batch @ torch.randn(32, 10)).shape) # applied to every token: (8, 16, 10)
assert (col + row).shape == (3, 4)

## 3. Why vectorize?

One tensor operation runs in optimized native code. The same arithmetic in a Python loop is far slower.

In [ ]:
a, b = torch.randn(200_000), torch.randn(200_000)

t0 = time.perf_counter(); slow = sum(a[i].item() * b[i].item() for i in range(len(a))); t_loop = time.perf_counter() - t0
t0 = time.perf_counter(); fast = (a * b).sum().item();                                   t_vec = time.perf_counter() - t0

print(f"loop {t_loop * 1000:.1f} ms, vectorized {t_vec * 1000:.3f} ms, about {t_loop / t_vec:,.0f}x faster")
assert abs(slow - fast) < 1.0

## 4. Autograd

Mark a tensor with `requires_grad=True` and PyTorch records every operation on it. `.backward()` then fills in `.grad`. This is the example from the [backprop lab](backprop.ipynb), and the answers match.

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(-3.0, requires_grad=True)
c = torch.tensor(10.0, requires_grad=True)
L = (a * b + c) * a
L.backward()
print("L =", L.item(), "| grads:", a.grad.item(), b.grad.item(), c.grad.item())
assert (a.grad.item(), b.grad.item(), c.grad.item()) == (-2.0, 4.0, 2.0)

from s2a.autograd import Value            # our own engine gives the same numbers
va, vb, vc = Value(2.0), Value(-3.0), Value(10.0)
((va * vb + vc) * va).backward()
assert (va.grad, vb.grad, vc.grad) == (-2.0, 4.0, 2.0)

Gradients **accumulate** across `backward()` calls, which is why training loops call `optimizer.zero_grad()` every step.

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
(w * w).backward(); print("after one backward:", w.grad.item())
(w * w).backward(); print("after two, without zeroing:", w.grad.item())
assert w.grad.item() == 12.0

## 5. Modules: weights plus a forward function

An `nn.Module` holds parameters and defines `forward`. Modules nest, and `.parameters()` finds every weight inside.

In [ ]:
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 4)     # 2*4 weights + 4 biases
        self.out = nn.Linear(4, 1)        # 4*1 weights + 1 bias

    def forward(self, x):
        return self.out(torch.tanh(self.hidden(x)))

model = TinyNet()
for name, p in model.named_parameters():
    print(f"{name:<14} {tuple(p.shape)}")
n_params = sum(p.numel() for p in model.parameters())
print("total:", n_params)
assert n_params == 17                     # same network as the backprop lab

## 6. The training loop

Five lines you'll see in every training script: forward, loss, zero the gradients, backward, step.

In [ ]:
X = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
Y = torch.tensor([[0.0], [1.0], [1.0], [0.0]])                 # XOR

optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
losses = []
for step in range(STEPS):
    pred = model(X)                         # forward
    loss = ((pred - Y) ** 2).mean()         # loss
    optimizer.zero_grad()                   # clear old gradients
    loss.backward()                         # backward
    optimizer.step()                        # update the weights
    losses.append(loss.item())

print(f"loss: {losses[0]:.4f} -> {losses[-1]:.6f} after {STEPS} steps")
with torch.no_grad():                       # no gradient tracking needed when just predicting
    print("predictions:", [round(v, 2) for v in model(X).squeeze(1).tolist()], "target: [0, 1, 1, 0]")
assert losses[-1] < losses[0]

## 7. Devices

The same code runs on a GPU if you move the model and the data there. Everything in one operation must be on the same device.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
out = model(X.to(device))
print("running on:", out.device)

## Try it

- Predict the shape before running: `torch.randn(5, 1, 3) + torch.randn(4, 1)`. Then one that fails: `torch.randn(3, 4) + torch.randn(3)`. Why?
- Remove `optimizer.zero_grad()` from the loop. What happens to training?
- Replace `Adam` with `torch.optim.SGD(model.parameters(), lr=0.2)`, the plain gradient descent from the backprop lab. How do the loss curves compare?
- Open `s2a/gpt.py` and find each idea from this lab: modules, shapes, the five-line loop.